In [4]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

# ----------------------
# Hyperparamètres
# ----------------------
GAMMA = 0.99
LR = 1e-4
BATCH_SIZE = 32
MEMORY_SIZE = 10000
EPS_START = 1.0
EPS_END = 0.01
EPS_DECAY = 50000
TARGET_UPDATE = 10

# ----------------------
# Réseau de neurones
# ----------------------
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# ----------------------
# Replay Buffer
# ----------------------
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

# ----------------------
# Fonction d'entraînement
# ----------------------
def train_dqn(env_name="CartPole-v1", episodes=200):

    env = gym.make(env_name)

    n_actions = env.action_space.n
    state_dim = env.observation_space.shape[0]

    policy_net = DQN(state_dim, n_actions)
    target_net = DQN(state_dim, n_actions)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(policy_net.parameters(), lr=LR)
    memory = ReplayBuffer(MEMORY_SIZE)

    epsilon = EPS_START
    steps_done = 0

    for ep in range(episodes):

        # -------- GYMNASIUM FIX --------
        state, _ = env.reset()
        # ------------------------------

        total_reward = 0
        done = False

        while not done:

            steps_done += 1

            # ε-greedy
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    state_tensor = torch.FloatTensor(state)
                    action = policy_net(state_tensor).argmax().item()

            # -------- GYMNASIUM FIX --------
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            # ------------------------------

            memory.push(state, action, reward, next_state, done)

            state = next_state
            total_reward += reward

            # epsilon decay
            epsilon = EPS_END + (EPS_START - EPS_END) * np.exp(-1. * steps_done / EPS_DECAY)

            # entraînement
            if len(memory) > BATCH_SIZE:

                states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

                states = torch.FloatTensor(states)
                actions = torch.LongTensor(actions)
                rewards = torch.FloatTensor(rewards)
                next_states = torch.FloatTensor(next_states)
                dones = torch.FloatTensor(dones)

                q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

                with torch.no_grad():
                    next_q_values = target_net(next_states).max(1)[0]
                    target_q_values = rewards + GAMMA * next_q_values * (1 - dones)

                loss = nn.MSELoss()(q_values, target_q_values)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        # mise à jour du target network
        if ep % TARGET_UPDATE == 0:
            target_net.load_state_dict(policy_net.state_dict())

        print(f"Episode {ep}, Reward: {total_reward:.2f}, Epsilon: {epsilon:.2f}")

    env.close()
    return policy_net


# ----------------------
# MAIN
# ----------------------
if __name__ == "__main__":
    trained_net = train_dqn("CartPole-v1", episodes=200)


Episode 0, Reward: 47.00, Epsilon: 1.00
Episode 1, Reward: 25.00, Epsilon: 1.00
Episode 2, Reward: 25.00, Epsilon: 1.00
Episode 3, Reward: 15.00, Epsilon: 1.00
Episode 4, Reward: 44.00, Epsilon: 1.00
Episode 5, Reward: 11.00, Epsilon: 1.00
Episode 6, Reward: 23.00, Epsilon: 1.00
Episode 7, Reward: 13.00, Epsilon: 1.00
Episode 8, Reward: 19.00, Epsilon: 1.00
Episode 9, Reward: 12.00, Epsilon: 1.00
Episode 10, Reward: 47.00, Epsilon: 0.99
Episode 11, Reward: 13.00, Epsilon: 0.99
Episode 12, Reward: 18.00, Epsilon: 0.99
Episode 13, Reward: 36.00, Epsilon: 0.99
Episode 14, Reward: 11.00, Epsilon: 0.99
Episode 15, Reward: 14.00, Epsilon: 0.99
Episode 16, Reward: 27.00, Epsilon: 0.99
Episode 17, Reward: 23.00, Epsilon: 0.99
Episode 18, Reward: 10.00, Epsilon: 0.99
Episode 19, Reward: 47.00, Epsilon: 0.99
Episode 20, Reward: 17.00, Epsilon: 0.99
Episode 21, Reward: 10.00, Epsilon: 0.99
Episode 22, Reward: 19.00, Epsilon: 0.99
Episode 23, Reward: 14.00, Epsilon: 0.99
Episode 24, Reward: 11.00,